Below is a merge that fits your current workflow and naming. It uses your existing:

    orbital_rotation_representation_R

    build_generalized_sectors

    evaluate_single_point

    run_scan

and adds the two energy-based indicators for the shared (a,b,c)(a,b,c) case in the same style as your current entropy / commutator diagnostics. The hooks are placed directly into evaluate_single_point, and the CSV fields are extended accordingly. This is aligned with the code structure in your current workflow and utility file.
1. Add these helper functions

Place them after build_generalized_sectors(...) and before if __name__ == "__main__": in your utility file.

In [1]:
import numpy as np

In [2]:


from dataclasses import dataclass

@dataclass
class EnergySectorDiagnostics:
    E_dec_min: float
    best_sector: object
    best_sector_dim: int
    E_coupled: float
    K_coupled: int
    coupled_converged: bool
    E_BO: float
    n_sectors: int


def diagonalize_sector_blocks(H_dense, sectors_dict):
    """
    Diagonalize each symmetry block H(s) independently.

    Returns
    -------
    sector_data : dict
        key -> {
            "idxs": list of determinant indices in this sector,
            "evals": ndarray of block eigenvalues,
            "evecs_full": list of full-space vectors (same basis ordering as H_dense)
        }
    """
    dim = H_dense.shape[0]
    sector_data = {}

    for key, idxs in sectors_dict.items():
        blk = H_dense[np.ix_(idxs, idxs)]
        blk = 0.5 * (blk + blk.conj().T)

        evals, evecs = np.linalg.eigh(blk)

        evecs_full = []
        for j in range(evecs.shape[1]):
            v = np.zeros(dim, dtype=np.complex128)
            v[np.asarray(idxs, dtype=int)] = evecs[:, j]
            evecs_full.append(v)

        sector_data[key] = {
            "idxs": idxs,
            "evals": evals,
            "evecs_full": evecs_full,
        }

    return sector_data


def decoupled_energy_test(H_dense, sectors_dict):
    """
    Decoupled-energy test:
        E_dec_min = min_s lambda_min(H(s))
    """
    best_E = None
    best_key = None
    best_dim = 0

    for key, idxs in sectors_dict.items():
        blk = H_dense[np.ix_(idxs, idxs)]
        blk = 0.5 * (blk + blk.conj().T)
        e0 = float(np.linalg.eigvalsh(blk)[0])

        if best_E is None or e0 < best_E:
            best_E = e0
            best_key = key
            best_dim = len(idxs)

    return best_E, best_key, best_dim


def coupled_energy_test(H_dense, sector_data, E_exact=None, tol=1e-8, max_total_vectors=None):
    """
    Coupled-energy test:
      1) collect low-energy eigenvectors from all sectors,
      2) sort by their block eigen-energies,
      3) project H into span of first K vectors,
      4) increase K until projected ground-state energy reaches E_exact within tol.

    If E_exact is None, uses all available vectors.
    """
    candidates = []
    for key, data in sector_data.items():
        for e, v in zip(data["evals"], data["evecs_full"]):
            candidates.append((float(e), key, v))

    candidates.sort(key=lambda t: t[0])

    if max_total_vectors is None:
        max_total_vectors = len(candidates)

    chosen_vecs = []
    chosen_keys = []
    E_proj = None
    K_final = 0
    converged = False

    for K in range(1, min(max_total_vectors, len(candidates)) + 1):
        e, key, v = candidates[K - 1]
        chosen_vecs.append(v)
        chosen_keys.append(key)

        V = np.column_stack(chosen_vecs)
        H_proj = V.conj().T @ H_dense @ V
        H_proj = 0.5 * (H_proj + H_proj.conj().T)

        evals_proj = np.linalg.eigvalsh(H_proj)
        E_proj = float(evals_proj[0])
        K_final = K

        if E_exact is not None and abs(E_proj - E_exact) <= tol:
            converged = True
            break

    return E_proj, K_final, converged, chosen_keys[:K_final]


def bo_like_coupled_energy_test(H_dense, sector_data):
    """
    BO-like coupled test:
      take one lowest-energy eigenvector from each sector,
      build projected Hamiltonian in their span,
      return its ground-state energy.
    """
    ordered = sorted(sector_data.items(), key=lambda kv: float(kv[1]["evals"][0]))

    vecs = []
    chosen_keys = []
    for key, data in ordered:
        vecs.append(data["evecs_full"][0])
        chosen_keys.append(key)

    V = np.column_stack(vecs)
    H_proj = V.conj().T @ H_dense @ V
    H_proj = 0.5 * (H_proj + H_proj.conj().T)

    evals_proj = np.linalg.eigvalsh(H_proj)
    return float(evals_proj[0]), len(vecs), chosen_keys


def shared_abc_energy_indicators(
    H_dense,
    basis_bitstrings,
    n_spatial,
    n_qubits,
    a,
    b,
    c,
    U_spatial=None,
    E_exact=None,
    tol=1e-8,
    label="",
):
    """
    Energy-based indicators for the shared-(a,b,c) workflow.

    If U_spatial is None:
        use H_dense directly and build sectors in the current determinant basis.

    If U_spatial is not None:
        rotate H into the orbital-rotated determinant basis using R,
        then build sectors there, matching your entropy workflow.
    """
    if E_exact is None:
        E_exact = float(np.linalg.eigvalsh(0.5 * (H_dense + H_dense.conj().T))[0])

    # Work in the rotated determinant basis if U is supplied
    if U_spatial is not None:
        R = orbital_rotation_representation_R(U_spatial, basis_bitstrings, n_spatial)
        H_work = R.conj().T @ H_dense @ R
        H_work = 0.5 * (H_work + H_work.conj().T)
    else:
        H_work = 0.5 * (H_dense + H_dense.conj().T)

    sectors = build_generalized_sectors(
        basis_bitstrings, n_spatial, n_qubits, a, b, c
    )

    sector_data = diagonalize_sector_blocks(H_work, sectors)

    E_dec_min, best_sector, best_sector_dim = decoupled_energy_test(H_work, sectors)
    E_coupled, K_coupled, converged, _ = coupled_energy_test(
        H_work, sector_data, E_exact=E_exact, tol=tol
    )
    E_BO, _, _ = bo_like_coupled_energy_test(H_work, sector_data)

    print(f"\n=== Energy indicators: {label} ===")
    print(f"E_exact            = {E_exact:+.12f}")
    print(f"E_dec_min          = {E_dec_min:+.12f}")
    print(f"best sector        = {best_sector}")
    print(f"best sector dim    = {best_sector_dim}")
    print(f"E_coupled          = {E_coupled:+.12f}")
    print(f"K_coupled          = {K_coupled}")
    print(f"coupled converged  = {converged}")
    print(f"E_BO               = {E_BO:+.12f}")
    print(f"n_sectors          = {len(sectors)}")

    return EnergySectorDiagnostics(
        E_dec_min=E_dec_min,
        best_sector=best_sector,
        best_sector_dim=best_sector_dim,
        E_coupled=E_coupled,
        K_coupled=K_coupled,
        coupled_converged=converged,
        E_BO=E_BO,
        n_sectors=len(sectors),
    )

# 2. Merge into evaluate_single_point(...)

# Add the following block inside evaluate_single_point, after you define:

# H_id = H_sub.toarray().astype(np.complex128)
# psi_id = v_sub / np.linalg.norm(v_sub)

# and before the final elapsed = time.time() - t0.
# Add this block:

#     # ============================================================
#     # Energy-based indicators (shared a,b,c case)
#     # ============================================================

#     energy_id = shared_abc_energy_indicators(
#         H_dense=H_id,
#         basis_bitstrings=basis_bitstrings,
#         n_spatial=n_spatial,
#         n_qubits=n_qubits,
#         a=a_id,
#         b=b_id,
#         c=c_id,
#         U_spatial=U_id,
#         E_exact=E_fci,
#         tol=1e-8,
#         label=f"{molecule} / identity",
#     )

#     energy_opt = shared_abc_energy_indicators(
#         H_dense=H_id,
#         basis_bitstrings=basis_bitstrings,
#         n_spatial=n_spatial,
#         n_qubits=n_qubits,
#         a=a_opt,
#         b=b_opt,
#         c=c_opt,
#         U_spatial=U_opt,
#         E_exact=E_fci,
#         tol=1e-8,
#         label=f"{molecule} / optimized",
#     )

# 3. Extend the printed summary

# Replace your current final print:

#     print(
#         f"[{molecule}] x={x:.4f} finished in {elapsed:.2f} s | "
#         f"V_opt={V_opt:.6f} | (a,b,c)=({a_opt:.4f},{b_opt:.4f},{c_opt:.4f})"
#     )

# with:

#     print(
#         f"[{molecule}] x={x:.4f} finished in {elapsed:.2f} s | "
#         f"V_opt={V_opt:.6f} | "
#         f"(a,b,c)=({a_opt:.4f},{b_opt:.4f},{c_opt:.4f}) | "
#         f"Edec_opt={energy_opt.E_dec_min:.6f} | "
#         f"Ecpl_opt={energy_opt.E_coupled:.6f} | "
#         f"EBO_opt={energy_opt.E_BO:.6f} | "
#         f"K_opt={energy_opt.K_coupled}"
#     )

# 4. Extend the returned row

# In evaluate_single_point, add these fields to the returned dict:

#         "Edec_Identity": energy_id.E_dec_min,
#         "Edec_Optimized": energy_opt.E_dec_min,
#         "Ecoupled_Identity": energy_id.E_coupled,
#         "Ecoupled_Optimized": energy_opt.E_coupled,
#         "Kcoupled_Identity": energy_id.K_coupled,
#         "Kcoupled_Optimized": energy_opt.K_coupled,
#         "EBO_Identity": energy_id.E_BO,
#         "EBO_Optimized": energy_opt.E_BO,
#         "NumSectors_Identity": energy_id.n_sectors,
#         "NumSectors_Optimized": energy_opt.n_sectors,

# So your full return { ... } becomes:

#     return {
#         "Molecule": molecule,
#         "Geometry_Param": x,
#         "E_FCI": E_fci,
#         "E_CISD": E_cisd,
#         "V_Identity": V_id,
#         "V_Optimized": V_opt,
#         "a": a_opt,
#         "b": b_opt,
#         "c": c_opt,
#         "Sum_CommSq_Identity": nc_id,
#         "Sum_CommSq_Optimized": nc_opt,
#         "Sum_Sexp_Identity": np.real(indiv_id["sum_exp"]) if EVAL_STATE_SPECIFIC_COMMUTATIVITY else np.nan,
#         "Sum_Sexp_Optimized": np.real(indiv_opt["sum_exp"]) if EVAL_STATE_SPECIFIC_COMMUTATIVITY else np.nan,
#         "Coarse_Entropy_Identity": I_SS_id,
#         "Coarse_Entropy_Optimized": I_SS_opt,
#         "Fine_Entropy_Identity": I_S_id,
#         "Fine_Entropy_Optimized": I_S_opt,
#         "Edec_Identity": energy_id.E_dec_min,
#         "Edec_Optimized": energy_opt.E_dec_min,
#         "Ecoupled_Identity": energy_id.E_coupled,
#         "Ecoupled_Optimized": energy_opt.E_coupled,
#         "Kcoupled_Identity": energy_id.K_coupled,
#         "Kcoupled_Optimized": energy_opt.K_coupled,
#         "EBO_Identity": energy_id.E_BO,
#         "EBO_Optimized": energy_opt.E_BO,
#         "NumSectors_Identity": energy_id.n_sectors,
#         "NumSectors_Optimized": energy_opt.n_sectors,
#     }

# 5. Extend CSV headers

# In run_scan(...), replace fieldnames = [...] with:

#     fieldnames = [
#         "Molecule",
#         "Geometry_Param",
#         "E_FCI",
#         "E_CISD",
#         "V_Identity",
#         "V_Optimized",
#         "a",
#         "b",
#         "c",
#         "Sum_CommSq_Identity",
#         "Sum_CommSq_Optimized",
#         "Sum_Sexp_Identity",
#         "Sum_Sexp_Optimized",
#         "Coarse_Entropy_Identity",
#         "Coarse_Entropy_Optimized",
#         "Fine_Entropy_Identity",
#         "Fine_Entropy_Optimized",
#         "Edec_Identity",
#         "Edec_Optimized",
#         "Ecoupled_Identity",
#         "Ecoupled_Optimized",
#         "Kcoupled_Identity",
#         "Kcoupled_Optimized",
#         "EBO_Identity",
#         "EBO_Optimized",
#         "NumSectors_Identity",
#         "NumSectors_Optimized",
#     ]

# 6. One important correction to your current block logic

# Your current entropy workflow already does the right conceptual thing for the optimized shared case:

# R_opt = orbital_rotation_representation_R(U_opt, basis_bitstrings, n_spatial)
# H_U = R_opt.conj().T @ (H_id @ R_opt)

# That is exactly the right basis for the energy-sector tests too. So for the new indicators, I reused the same R_opt logic rather than introducing a separate Fock-rotation implementation. This makes the energy tests consistent with your existing shared-(a,b,c)(a,b,c) entropy pipeline.
# 7. Small bug fix worth making

# In your existing helper:

# def decoupled_min_energy(H, sectors):
#     Emin = None
#     for idxs in sectors.values():
#         blk = H_dense[np.ix_(idxs, idxs)]

# this uses H_dense instead of the function argument H. That is a bug. Either delete that helper or fix it to:

def decoupled_min_energy(H, sectors):
    Emin = None
    for idxs in sectors.values():
        blk = H[np.ix_(idxs, idxs)]
        e0 = float(np.linalg.eigvalsh(blk)[0])
        Emin = e0 if Emin is None else min(Emin, e0)
    return float(Emin)

The new implementation above already replaces it.
8. Minimal merged version of the new code path

If you want the shortest view of the merge, the new workflow inside evaluate_single_point is:

H_id = H_sub.toarray().astype(np.complex128)

energy_id = shared_abc_energy_indicators(
    H_dense=H_id,
    basis_bitstrings=basis_bitstrings,
    n_spatial=n_spatial,
    n_qubits=n_qubits,
    a=a_id, b=b_id, c=c_id,
    U_spatial=U_id,
    E_exact=E_fci,
    label=f"{molecule} / identity",
)

energy_opt = shared_abc_energy_indicators(
    H_dense=H_id,
    basis_bitstrings=basis_bitstrings,
    n_spatial=n_spatial,
    n_qubits=n_qubits,
    a=a_opt, b=b_opt, c=c_opt,
    U_spatial=U_opt,
    E_exact=E_fci,
    label=f"{molecule} / optimized",
)

SyntaxError: invalid syntax (2456179211.py, line 211)